In [1]:
import json
from copy import deepcopy
from tqdm import tqdm
import re
import os
import random

In [2]:
# Load datasets accordingly to the subtask

SUBTASK = "subtask 2"
# Select which file we should be working on
file_to_load = f"../data/SOMD 2026/{SUBTASK}/train_data.jsonl"
with open(file_to_load, 'r') as f:
    train_data = [json.loads(l) for l in list(f)]

with open(f"../data/SOMD 2026/{SUBTASK}/train_labels.json", "r") as f:
    train_labels = json.load(f)

print(len(train_data))
print(len(train_labels))

2860
699


In [ ]:
# percentage = [0.25, 0.50, 0.75, 1.0]

# for noise_rate in percentage:
#     random.seed(42)

#     new_train_data = deepcopy(train_data)

#     n_to_modify = int(len(new_train_data) * noise_rate)
#     indices_to_modify = set(random.sample(range(len(new_train_data)), n_to_modify))


#     # Creating a new set with one word added at the end of the original mention
#     for i in tqdm(indices_to_modify):
#         e = new_train_data[i]
#         # print(f"Original mention: {e['mention']}")



#         # if we want to add right noise
#         # try:
#         #     next_word = e["sentence"][e["end"]:].split()[0]
#         #     e["end"] = e["end"] + len(next_word) + 1
#         #     e["mention"] = e["sentence"][e["start"]:e["end"]]
#         # except:
#         #     print("bug")
#         #print(f"New mention: {e["mention"]}")

#         # if we want to add left noise
#         # prefix = e["sentence"][:e["start"]].rstrip().split()
#         # if not prefix:
#         #     continue
#         # prev_word = prefix[-1]
#         # e["start"] -= len(prev_word) + 1
#         # e["mention"] = e["sentence"][e["start"]:e["end"]]
#         # print(f"New mention: {e['mention']}")


#         # If we want to add Left + Right 
#         # left side
#         prefix = e["sentence"][:e["start"]].rstrip().split()
#         if prefix:
#             prev_word = prefix[-1]
#             e["start"] -= len(prev_word) + 1

#         # right side
#         suffix = e["sentence"][e["end"]:].lstrip().split()
#         if suffix:
#             next_word = suffix[0]
#             e["end"] += len(next_word) + 1

#         e["mention"] = e["sentence"][e["start"]:e["end"]]


#         # print(e["sentence"])
#         # print("-------------------------------------------------")

#     filename = f"../data/SOMD 2026/{SUBTASK}/noisy_data/left_right_noise/left_right_noise_{noise_rate}_train_data.jsonl"
#     with open(filename, 'w') as f:
#         for item in new_train_data:
#             json_record = json.dumps(item)
#             f.write(json_record + '\n')

100%|██████████| 2860/2860 [00:00<00:00, 512834.40it/s]


In [3]:
def inject_boundary_noise(train_data, noise_rate):
    """
    Add boundary errors with different error types.
    
    Error types:
    - Add word on right (40%)
    - Add word on left (30%)
    - Add words on both sides (15%)
    - Truncate right (10%)
    - Truncate left (5%)
    """
    new_train_data = deepcopy(train_data)
    
    import random
    n_to_modify = int(len(new_train_data) * noise_rate)
    indices_to_modify = set(random.sample(range(len(new_train_data)), n_to_modify))
    
    stats = {'add_right': 0, 
             'add_left': 0, 
             'add_both': 0, 
             'truncate_right': 0, 
             'truncate_left': 0, 
             'skipped': 0}
    
    for idx, mention in enumerate(tqdm(new_train_data, desc="Injecting boundary noise")):
        if idx not in indices_to_modify:
            continue
        
        sentence = mention["sentence"]
        start = mention["start"]
        end = mention["end"]
        current_mention = mention["mention"]
        
        # Randomly choose error type
        error_type = random.choices(
            ['add_right', 'add_left', 'add_both', 'truncate_right', 'truncate_left'],
            weights=[40, 30, 15, 10, 5]
        )[0]
        
        try:
            if error_type == 'add_right':
                # Add next word
                after_text = sentence[end:].strip()
                if after_text:
                    match = re.match(r"^(\s*)(\S+)", sentence[end:])
                    if match:
                        new_end = end + len(match.group(0))
                        mention["end"] = new_end
                        mention["mention"] = sentence[start:new_end]
                        stats['add_right'] += 1
                    else:
                        stats['skipped'] += 1
                else:
                    stats['skipped'] += 1
            
            elif error_type == 'add_left':
                # Add previous word
                before_text = sentence[:start].strip()
                if before_text:
                    match = re.search(r"(\S+)(\s*)$", sentence[:start])
                    if match:
                        prev_word_start = match.start()
                        mention["start"] = prev_word_start
                        mention["mention"] = sentence[prev_word_start:end]
                        stats['add_left'] += 1
                    else:
                        stats['skipped'] += 1
                else:
                    stats['skipped'] += 1
            
            elif error_type == 'add_both':
                # Add both previous and next word
                # (combine add_left and add_right logic)
                before = sentence[:start].strip()
                after = sentence[end:].strip()
                if before and after:
                    left_match = re.search(r"(\S+)(\s*)$", sentence[:start])
                    right_match = re.match(r"^(\s*)(\S+)", sentence[end:])
                    if left_match and right_match:
                        new_start = left_match.start()
                        new_end = end + len(right_match.group(0))
                        mention["start"] = new_start
                        mention["end"] = new_end
                        mention["mention"] = sentence[new_start:new_end]
                        stats['add_both'] += 1
                    else:
                        stats['skipped'] += 1
                else:
                    stats['skipped'] += 1
            
            elif error_type == 'truncate_right':
                # Remove last 1-2 characters
                if len(current_mention) > 3:
                    chars_to_remove = random.randint(1, min(2, len(current_mention) - 2))
                    mention["end"] = end - chars_to_remove
                    mention["mention"] = sentence[start:end - chars_to_remove]
                    stats['truncate_right'] += 1
                else:
                    stats['skipped'] += 1
            
            elif error_type == 'truncate_left':
                # Remove first 1-2 characters
                if len(current_mention) > 3:
                    chars_to_remove = random.randint(1, min(2, len(current_mention) - 2))
                    mention["start"] = start + chars_to_remove
                    mention["mention"] = sentence[start + chars_to_remove:end]
                    stats['truncate_left'] += 1
                else:
                    stats['skipped'] += 1
        
        except Exception as e:
            stats['skipped'] += 1
            continue
    
    print(f"\nBoundary noise injection stats:")
    for error_type, count in stats.items():
        print(f"  {error_type}: {count}")
    
    total_modified = sum(stats.values()) - stats['skipped']
    print(f"  Total modified: {total_modified}/{len(new_train_data)} ({total_modified/len(new_train_data):.1%})")
    
    return new_train_data


In [4]:
noise_rates = [0, 0.25, 0.50, 0.75, 1.0]

for noise_rate in noise_rates:
    noisy_data = inject_boundary_noise(train_data, noise_rate = noise_rate)
    # Save dataset locally 
    filename = f"../../SOMD-2026/data/SOMD 2026/subtask 2/noisy_data_boundary/noisy_{noise_rate}_train_data_subtask_2.jsonl"
    with open(filename, 'w') as f:
        for item in noisy_data:
            json_record = json.dumps(item)
            f.write(json_record + '\n')

Injecting boundary noise: 100%|██████████| 2860/2860 [00:00<00:00, 7487958.45it/s]



Boundary noise injection stats:
  add_right: 0
  add_left: 0
  add_both: 0
  truncate_right: 0
  truncate_left: 0
  skipped: 0
  Total modified: 0/2860 (0.0%)


Injecting boundary noise: 100%|██████████| 2860/2860 [00:00<00:00, 793681.98it/s]



Boundary noise injection stats:
  add_right: 287
  add_left: 174
  add_both: 109
  truncate_right: 71
  truncate_left: 35
  skipped: 39
  Total modified: 676/2860 (23.6%)


Injecting boundary noise: 100%|██████████| 2860/2860 [00:00<00:00, 420238.55it/s]



Boundary noise injection stats:
  add_right: 575
  add_left: 380
  add_both: 194
  truncate_right: 139
  truncate_left: 63
  skipped: 79
  Total modified: 1351/2860 (47.2%)


Injecting boundary noise: 100%|██████████| 2860/2860 [00:00<00:00, 280824.74it/s]



Boundary noise injection stats:
  add_right: 851
  add_left: 592
  add_both: 342
  truncate_right: 170
  truncate_left: 89
  skipped: 101
  Total modified: 2044/2860 (71.5%)


Injecting boundary noise: 100%|██████████| 2860/2860 [00:00<00:00, 219524.73it/s]


Boundary noise injection stats:
  add_right: 1199
  add_left: 758
  add_both: 401
  truncate_right: 238
  truncate_left: 112
  skipped: 152
  Total modified: 2708/2860 (94.7%)


Injecting boundary noise: 100%|██████████| 2860/2860 [00:00<00:00, 7744163.62it/s]



Boundary noise injection stats:
  add_right: 0
  add_left: 0
  add_both: 0
  truncate_right: 0
  truncate_left: 0
  skipped: 0
  Total modified: 0/2860 (0.0%)


Injecting boundary noise: 100%|██████████| 2860/2860 [00:00<00:00, 698114.96it/s]



Boundary noise injection stats:
  add_right: 277
  add_left: 196
  add_both: 96
  truncate_right: 65
  truncate_left: 36
  skipped: 45
  Total modified: 670/2860 (23.4%)


Injecting boundary noise: 100%|██████████| 2860/2860 [00:00<00:00, 422562.68it/s]



Boundary noise injection stats:
  add_right: 553
  add_left: 410
  add_both: 204
  truncate_right: 147
  truncate_left: 61
  skipped: 55
  Total modified: 1375/2860 (48.1%)


Injecting boundary noise: 100%|██████████| 2860/2860 [00:00<00:00, 307905.99it/s]



Boundary noise injection stats:
  add_right: 855
  add_left: 566
  add_both: 300
  truncate_right: 200
  truncate_left: 99
  skipped: 125
  Total modified: 2020/2860 (70.6%)


Injecting boundary noise: 100%|██████████| 2860/2860 [00:00<00:00, 219039.70it/s]


Boundary noise injection stats:
  add_right: 1118
  add_left: 815
  add_both: 393
  truncate_right: 263
  truncate_left: 124
  skipped: 147
  Total modified: 2713/2860 (94.9%)
